# 2. Embeddings: make the chunks searchable by meaning

A keyword search can't match *"How much money did Apple make?"* with a chunk that says *"total net sales"*.
**Embeddings** solve this: they turn text into a list of numbers that represents its **meaning**.
Texts with similar meaning get similar numbers, even when they use different words.

- **Embedding model:** `all-MiniLM-L6-v2`, free, runs locally on your laptop, 384 numbers per text.
- **Vector database:** Chroma stores every chunk's text, embedding and labels, and finds the chunks closest in meaning to a question.

## Setup

In [1]:
from pathlib import Path

# Project root = the folder that contains requirements.txt (works no matter where the notebook runs from)
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("Project folder:", ROOT)

Project folder: /Users/os_bhosale/Desktop/Projects/Financial-Filings-Rag


In [2]:
import shutil
import numpy as np
import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14411.49it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The first run downloads the model (~90 MB). A warning about *unauthenticated requests to the HF Hub* can be ignored.

## Step 1: See embeddings in action

**Cosine similarity** measures how close two meanings are: 1.0 = same meaning, 0 = unrelated.

In [3]:
sentences = [
    "Apple's revenue grew strongly this year.",
    "Apple's sales increased compared to last year.",
    "Tesla is building a new factory in Texas.",
]
vectors = np.array(embedding_model.embed_documents(sentences))
print("Shape:", vectors.shape, "(3 sentences x 384 numbers)")


def similarity(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


print("Sentence 1 vs 2:", round(similarity(vectors[0], vectors[1]), 2))
print("Sentence 1 vs 3:", round(similarity(vectors[0], vectors[2]), 2))

Shape: (3, 384) (3 sentences x 384 numbers)
Sentence 1 vs 2: 0.81
Sentence 1 vs 3: 0.18


Sentences 1 and 2 share almost no words but score high, because they **mean** the same thing. Sentence 3 is about something else.

## Step 2: Build the search index

Chroma embeds all chunks and saves them to the `chroma_db` folder.
Any old index is deleted first: building on top of an existing one would store every chunk twice.

In [4]:
CHROMA_DIR = ROOT / "chroma_db"
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

chunks = pd.read_csv(ROOT / "data" / "chunks.csv")
documents = [
    Document(page_content=row.text,
             metadata={"ticker": row.ticker, "section": row.section, "chunk_id": row.chunk_id})
    for row in chunks.itertuples()
]

vector_db = Chroma.from_documents(
    documents,
    embedding_model,
    ids=chunks["chunk_id"].tolist(),
    persist_directory=str(CHROMA_DIR),
    collection_name="filings",
)
print("Chunks in the database:", vector_db._collection.count())

Chunks in the database: 1633


## Step 3: Test the search

`distance` = how far a chunk is from the question in "meaning space". **Lower = more similar.**

In [5]:
def show_search(question, k=3, **kwargs):
    print("Q:", question)
    for doc, distance in vector_db.similarity_search_with_score(question, k=k, **kwargs):
        print(f"  {distance:.3f} | {doc.metadata['chunk_id']:9s} | {doc.metadata['section'][:40]:40s} | "
              + doc.page_content[:70].replace("\n", " "))
    print()


show_search("What were Apple's total net sales in 2026?")
show_search("How many employees does Tesla have?")

Q: What were Apple's total net sales in 2026?
  0.581 | AAPL_139  | Item 7. Management's Discussion and Anal | 2026 | Change | 2025 | Change | 2024 iPhone | 254,098 | 21 | 209,586 |
  0.592 | AAPL_25   | Item 1. Business                         | Apple Inc. | 2026 Form 10-K | 4 Business Seasonality and Product Intro
  0.680 | AAPL_168  | Item 8. Financial Statements             | 2026 | 2025 | 2024 iPhone | 254,098 | 209,586 | 201,183 Mac | 37,929 |

Q: How many employees does Tesla have?
  0.577 | TSLA_65   | Item 1. Business                         | Tesla’s ability to attract, develop and retain a talented workforce is
  0.958 | TSLA_4    | Cover page                               | TESLA, INC. ANNUAL REPORT ON FORM 10-K FOR THE YEAR ENDED DECEMBER 31,
  0.968 | TSLA_67   | Item 1. Business                         | Tesla is committed to providing a workplace where employees feel respe



Every chunk has labels, so the search can be limited to one company with a **filter**.
A question that names no company searches everything; the filter restricts it to one company's report.

In [6]:
show_search("What was the total revenue last year?")
show_search("What was the total revenue last year?", filter={"ticker": "MSFT"})

Q: What was the total revenue last year?
  0.698 | NVDA_176  | Item 7. Management's Discussion and Anal | Year Ended Jan 25, 2026 | Jan 26, 2025 | Change ($ in millions, except
  0.755 | TSLA_192  | Item 7. Management's Discussion and Anal | Results of Operations Revenues
  0.766 | NVDA_306  | Item 15. Exhibits and Schedules          | Compute & Networking | Graphics | Total (In millions) Year Ended Jan 2

Q: What was the total revenue last year?
  0.828 | MSFT_312  | Item 8. Financial Statements             | Segment revenue, cost of revenue, operating expenses, and operating in
  0.893 | MSFT_313  | Item 8. Financial Statements             | (In millions) Year Ended June 30, | 2026 | 2025 | 2024 Productivity an
  0.921 | MSFT_294  | Item 8. Financial Statements             | 76  PART II Item 8  We are subject to income tax in many jurisdictions



**Output:** the `chroma_db` folder, the search index used by the RAG pipeline and the app.

Next: `3_rag/rag_demo.ipynb` connects the search to the LLM.